In [1]:
import pandas as pd
df = pd.read_pickle('../output/df_preprocessed.pkl')
df['processed_text'] = df['processed_tokens'].apply(lambda x: ' '.join(x))

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['processed_text'])
print(f"TF-IDF Matrix Form: {tfidf_matrix.shape}")

TF-IDF Matrix Form: (2540, 1063)


In [3]:
from gensim.models import Word2Vec
import numpy as np

w2v_model = Word2Vec(df['processed_tokens'].tolist(), vector_size=100, window=5, min_count=5, workers=4, epochs=10)
print(f"Word2Vec Vokabular-Größe: {len(w2v_model.wv.key_to_index)}")

def document_vector(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv.key_to_index]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

w2v_doc_vectors = np.array([document_vector(t, w2v_model) for t in df['processed_tokens']])
print(f"Word2Vec Dokument-Vektoren Form: {w2v_doc_vectors.shape}")

Word2Vec Vokabular-Größe: 941
Word2Vec Dokument-Vektoren Form: (2540, 100)


In [4]:
print("Wichtigste TF-IDF-Begriffe insgesamt:", tfidf.get_feature_names_out()[:15])

try:
    aehnliche = w2v_model.wv.most_similar('muell', topn=10)
    print("Wörter, die Word2Vec als ähnlich zu 'muell' einstuft:", aehnliche)
except KeyError:
    print("Das Wort 'muell' kam zu selten vor, probiere ein anderes Wort aus deinen Daten.")

dichte = tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])
print(f"Anteil der befüllten Zellen in der TF-IDF-Matrix: {dichte:.4%} (bei Word2Vec sind es nahezu 100 %)")

Wichtigste TF-IDF-Begriffe insgesamt: ['abbauen' 'abbrechen' 'abdeckung' 'abends' 'abfall' 'abgelade'
 'abgelaufen' 'abgemeldet' 'abgemeldet auto' 'abgemeldet fahrzeug'
 'abgemeldeter' 'abgestellt' 'abhilfe' 'abholen' 'abholung']
Wörter, die Word2Vec als ähnlich zu 'muell' einstuft: [('alt', 0.9995332956314087), ('direkt', 0.99947589635849), ('offensichtlich', 0.9994668960571289), ('einfach', 0.9994358420372009), ('Richtung', 0.999435544013977), ('ueber', 0.9994170069694519), ('Ecke', 0.9994141459465027), ('Stadt', 0.9993973970413208), ('Schild', 0.999392569065094), ('bitte', 0.9993844032287598)]
Anteil der befüllten Zellen in der TF-IDF-Matrix: 0.6309% (bei Word2Vec sind es nahezu 100 %)


In [5]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(max_features=3000, min_df=5, max_df=0.8, ngram_range=(1, 2))
count_matrix = count_vectorizer.fit_transform(df['processed_text'])

In [6]:
import pickle
with open('../output/vektoren.pkl', 'wb') as f:
    pickle.dump({'tfidf': tfidf, 'tfidf_matrix': tfidf_matrix, 'count_vectorizer': count_vectorizer, 'count_matrix': count_matrix}, f)
df.to_pickle('../output/df_vektorisiert.pkl')
print("Gespeichert.")


Gespeichert.
